# Swimplaces playground

Scratch notebook pro rychle overeni parseru a importeru nad `source_data/sample_data.csv`.

In [8]:
import os
import sys
from pathlib import Path

ROOT_DIR = Path.cwd()
if ROOT_DIR.name == "src":
    ROOT_DIR = ROOT_DIR.parent

SRC_DIR = ROOT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")

ROOT_DIR

PosixPath('/mnt/c/Dev/COex/coex_task/coex_swimplaces')

In [9]:
import django
from django.core.management import call_command

django.setup()
call_command("migrate", verbosity=0, interactive=False)

SAMPLE_CSV = ROOT_DIR / "source_data" / "sample_data.csv"
SAMPLE_CSV

SynchronousOnlyOperation: You cannot call this from an async context - use a thread or sync_to_async.

## Nacteni 10 vzorku

In [ ]:
import csv

with SAMPLE_CSV.open(encoding="utf-8-sig", newline="") as sample_file:
    reader = csv.reader(sample_file, delimiter=";")
    header = next(reader)
    sample_rows = list(reader)

len(sample_rows), header

(10,
 ['MapoticID',
  'Longitude',
  'Latitude',
  'Name',
  'Category',
  'Rating',
  'Image URL',
  'Import ID',
  'Description',
  'Address',
  'Web',
  'E-mail',
  'Phone number',
  'Description',
  'Refreshment',
  'Diving',
  'Entrance',
  'Accessibility/parking',
  'Link',
  'Nudist beach',
  'Video',
  'Dog swimming'])

## Parser preview

Ukazka mapovani vzorku do import DTO.

In [ ]:
from places.services.parsers import parse_swimplace_row

parsed_rows = [parse_swimplace_row(row) for row in sample_rows]

[
    {
        "external_id": row.external_id,
        "import_id": row.import_id,
        "name": row.name,
        "category": row.category,
        "rating": row.rating,
        "dog_swimming": row.dog_swimming,
    }
    for row in parsed_rows
    if row is not None
]

[{'external_id': 202693,
  'import_id': 'swimplaces:1',
  'name': 'Mlékojedy (Free D Bar)',
  'category': 'Sand quarry',
  'rating': Decimal('3.48'),
  'dog_swimming': None},
 {'external_id': 202694,
  'import_id': 'swimplaces:2',
  'name': 'Jezero Lhota',
  'category': 'Sand quarry',
  'rating': Decimal('4.40'),
  'dog_swimming': None},
 {'external_id': 202695,
  'import_id': 'swimplaces:3',
  'name': 'Pískovna Konětopy',
  'category': 'Sand quarry',
  'rating': Decimal('4.33'),
  'dog_swimming': None},
 {'external_id': 202696,
  'import_id': 'swimplaces:4',
  'name': 'Jezero Ovčáry',
  'category': 'Lake (pond)',
  'rating': Decimal('3.73'),
  'dog_swimming': None},
 {'external_id': 202697,
  'import_id': 'swimplaces:5',
  'name': 'Beroun - Lom Kosov',
  'category': 'Quarry',
  'rating': Decimal('4.13'),
  'dog_swimming': None},
 {'external_id': 202698,
  'import_id': 'swimplaces:6',
  'name': 'Lomy u Prosetína',
  'category': 'Quarry',
  'rating': None,
  'dog_swimming': None},
 {'ex

## Import sample dat

Importer uklada do standardni lokalni DB. Druhe spusteni ma byt idempotentni: `created=0`, `updated=10`, pokud uz sample data existuji.

In [ ]:
from places.models import SwimPlace
from places.services.swimplace_importer import import_swim_places

first_summary = import_swim_places(SAMPLE_CSV)
second_summary = import_swim_places(SAMPLE_CSV)

{
    "first_run": first_summary,
    "second_run": second_summary,
    "total_db_count": SwimPlace.objects.count(),
}

## Kontrola ulozenych sample dat

In [ ]:
sample_external_ids = [row.external_id for row in parsed_rows if row is not None]

list(
    SwimPlace.objects.filter(external_id__in=sample_external_ids)
    .order_by("external_id")
    .values(
        "external_id",
        "import_id",
        "name",
        "category",
        "rating",
        "dog_swimming",
    )
)